# Importing libraries

In [1]:
# install packages in colab if necessary
#!pip install nlpaug tdqm sacremoses

from datasets import load_dataset
import nlpaug.augmenter.word as naw
from transformers import MarianMTModel, MarianTokenizer
import pandas as pd
import torch
import gc
from tqdm import tqdm
import re

tqdm.pandas()


rseed = 42

# Importing HuggingFace Token

In [2]:
from huggingface_hub import login
from getpass import getpass

hftoken = getpass("HF-Token ")
login(token=hftoken)

# Masking the MBTI labels in text data
Since the backtranslation model tried to translate the MBTI labels frequently, usually resulting in them being chopped up or similar, there needs to be a placeholder to prevent this. Funnily enough, using a short name like "John" works almost perfectly.

In [ ]:
# this is a placeholder, chosen based on it not being translated nor chopped up
PLACEHOLDER = "John"

def protect_mbti_mask(text):
    return text.replace("<mbti>", PLACEHOLDER)

def restore_mbti_mask(text):
    return re.sub(rf"{PLACEHOLDER}", "<mbti>", text, flags=re.IGNORECASE)


# Oversampling
## Creating the functions for the oversampling processes
I am using synonymisation, backtranslation, random swapping and random deletion.

In [ ]:
# Synonym Augmenter
# prob 0.4
def synonymizer(text, aug1):
    protected = protect_mbti_mask(text)
#    coin = np.random.rand()
#    if coin < 0.4:
    augmented_text = aug1.augment(protected)
    augmented_text = augmented_text[0] if isinstance(augmented_text, list) else augmented_text
    return restore_mbti_mask(augmented_text)
#    else:
#        augmented_text = aug2.augment(text)
#        return augmented_text[0] if isinstance(augmented_text, list) else augmented_text

In [ ]:
# backtranslation
# prob 0.2
# def back_translation(text, aug):
#     augmented_text = aug.augment(text)
#     return augmented_text[0] if isinstance(augmented_text, list) else augmented_text
def backtranslate_safe(texts, device="cuda"):
    en_de_tok = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-de")
    en_de_mod = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-de").to(device)
    de_en_tok = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-de-en")
    de_en_mod = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-de-en").to(device)

    gen_kwargs = dict(
        num_beams=4,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
        max_length=512,
        early_stopping=True,
    )

    def translate(model, tokenizer, batch):
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                          truncation=True, max_length=512).to(device)
        with torch.no_grad():
            out = model.generate(**inputs, **gen_kwargs)
        return tokenizer.batch_decode(out, skip_special_tokens=True)

    protected = [protect_mbti_mask(t) for t in texts]
    results = []
    for i in range(0, len(protected), 16):
        batch = protected[i:i+16]
        de = translate(en_de_mod, en_de_tok, batch)
        back = translate(de_en_mod, de_en_tok, de)
        results.extend(back)

    del en_de_mod, de_en_mod
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return [restore_mbti_mask(t) for t in results]





In [ ]:
# random swap
# prob 0.2
def random_swap(text, aug):
    augmented_text = aug.augment(text)
    return augmented_text[0] if isinstance(augmented_text, list) else augmented_text


#  random deletion
# prob 0.2
def random_del(text, aug):
    augmented_text = aug.augment(text)
    return augmented_text[0] if isinstance(augmented_text, list) else augmented_text

## Loading the data
Need to chooose the data source here. Default is on the Alvis cluster, that was used to fine-tune the model later as well.

In [5]:
# data local
#df = pd.read_csv("..\data\csv\mbti_cleaned.csv")

# data colab
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv("/content/drive/MyDrive/Data/mbti_cleaned.csv")

#data alvis
#df = pd.read_csv("~/MA/data/mbti_cleaned.csv")

# huggingface
df_hfdict = load_dataset("DrinkIcedT/mbti")
df_pd = {split: ds.to_pandas() for split, ds in df_hfdict.items()}

df = df_pd["train"]


## Data preparation
First subsets of those labels I need to oversample are created. In that I also determine the share of each oversampling method of the whole amount of oversampled observations:

70% Backtranslation
20% Synonymisation
5% Random Swapping
5% Random Deletion

The portions are decided based on augmentation quality and reliability. 

In [ ]:
def label_samples(df, label, n_goal, rseed):
    obs_count = (df["label"] == label).sum()
    if obs_count < n_goal:
        df_subset = df[df["label"] == label]

        n_diff = n_goal - obs_count

        n_syn = round(n_diff*0.2)
        n_bt = round(n_diff*0.7)
        n_sw = round(n_diff*0.05)
        n_del = n_diff - (n_syn+n_bt+n_sw)



        syn_sample = df_subset.sample(n=n_syn, replace=True,random_state=rseed)
        bt_sample = df_subset.sample(n=n_bt, replace=True,random_state=rseed)
        sw_sample = df_subset.sample(n=n_sw, replace=True,random_state=rseed)
        del_sample = df_subset.sample(n=n_del, replace=True,random_state=rseed)

        return syn_sample, bt_sample, sw_sample, del_sample
    else:
        empty = df.iloc[:0]
        return empty, empty, empty, empty

In [ ]:
syn_sample = pd.DataFrame({"label": [None], "post": [None]}, columns = ["label", "post"])
bt_sample = pd.DataFrame({"label": [None], "post": [None]}, columns = ["label", "post"])
sw_sample = pd.DataFrame({"label": [None], "post": [None]}, columns = ["label", "post"])
del_sample = pd.DataFrame({"label": [None], "post": [None]}, columns = ["label", "post"])

for label in df["label"].unique():
    s1, s2, s3, s4 = label_samples(df, label, n_goal=10000, rseed=42)
    

    syn_sample = pd.concat([syn_sample, s1]).dropna()
    bt_sample = pd.concat([bt_sample, s2]).dropna()
    sw_sample = pd.concat([sw_sample, s3]).dropna()
    del_sample = pd.concat([del_sample, s4]).dropna()


    



## Augmentation pipeline

In [ ]:
def data_augmenter(syn_sample, bt_sample, sw_sample, del_sample):
    #device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
    
    #aug_syn_word = naw.SynonymAug(aug_src = "wordnet")
    aug_syn_bert =naw.ContextualWordEmbsAug(
        model_path='bert-base-uncased',
        action="substitute",
        device="cuda"
    )
    # can also add wordnet if wanted
    syn_sample["post_augmented"] = syn_sample["post"].progress_apply(synonymizer, args=(aug_syn_bert,))

    # free up vram
    #del aug_syn_word
    del aug_syn_bert

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # aug_backtranslation = naw.BackTranslationAug(
    #     from_model_name="facebook/wmt19-en-de",
    #     to_model_name="facebook/wmt19-de-en",
    #     device= "cuda",
    #     batch_size=16
    # )

    texts = bt_sample["post"].tolist()
    bt_sample["post_augmented"] = backtranslate_safe(texts)

    # free up vram
    # del aug_backtranslation
    # gc.collect()

    # if torch.cuda.is_available():
    #     torch.cuda.empty_cache()


    aug_rand_swap = naw.RandomWordAug(action = "swap")
    sw_sample["post_augmented"] = sw_sample["post"].progress_apply(random_swap, args = (aug_rand_swap,))
    
    aug_rand_del = naw.RandomWordAug(action = "delete")
    del_sample["post_augmented"] = del_sample["post"].progress_apply(random_del, args = (aug_rand_del,))


    return syn_sample,bt_sample,sw_sample,del_sample



In [ ]:
# fixing nltk dependency
import nltk
#nltk.download('averaged_perceptron_tagger_eng')

# run data_augmenter
syn_df, bt_df, sw_df, del_df = data_augmenter(syn_sample, bt_sample, sw_sample, del_sample)

In [ ]:
#saving data
# syn_df.to_csv("syn_augmented.csv", sep='\t', index=False, quoting=1)
# bt_df.to_csv("bt_augmented.csv", sep='\t', index=False, quoting=1)
# sw_df.to_csv("sw_augmented.csv", sep='\t', index=False, quoting=1)
# del_df.to_csv("del_augmented.csv", sep='\t', index=False, quoting=1)
# print("Alle Dateien erfolgreich als CSV gespeichert!")
import pandas as pd
# reading finished datasets locally, for further processing and metrics, so the augmentation process is not needed again
# syn_df = pd.read_csv("../data/csv/latest/syn_augmented_ver4.csv", sep = "\t", quoting = 1)
# bt_df = pd.read_csv("../data/csv/latest/bt_augmented_john.csv", sep = "\t", quoting = 1)
# sw_df = pd.read_csv("../data/csv/latest/sw_augmented_ver3.csv", sep = "\t", quoting = 1)
# del_df = pd.read_csv("../data/csv/latest/del_augmented_ver3.csv", sep = "\t", quoting = 1)

bt_df.head()

In [ ]:
sw_df.info()
del_df.info()
bt_df.info()
syn_df.info()


## Evaluating MBTI label masking


In [ ]:
# only one errornous observation
syn_df["post_augmented"].iloc[7788]

In [ ]:
print(f"bt_df mbti mask counts in post col (pre aug): {bt_df["post"].apply(lambda x: "<mbti>" in x).sum()}")
print(f"bt_df mbti mask counts in augmented col: {bt_df["post_augmented_john"].apply(lambda x: "<mbti>" in x).sum()}")
#print(f"bt_df failed mbti masking example, counts in augmented col: {bt_df["post_augmented"].apply(lambda x: "MBTITOPOKENPLACECCHOLDER" in x).sum()}")


print(f"syn_df mbti mask counts in post col (pre aug): {syn_df["post"].apply(lambda x: "<mbti>" in x).sum()}")
print(f"syn_df mbti mask counts in augmented col: {syn_df["post_augmented"].apply(lambda x: "<mbti>" in x).sum()}")
print(f"syn_df UNK mask counts in augmented col: {syn_df["post_augmented"].apply(lambda x: "UNK" in x).sum()}")
print(f"syn_df john mask counts in post col (pre aug): {syn_df["post"].apply(lambda x: "john" in x).sum()}")


print(f"sw_df mbti mask counts in post col (pre aug): {sw_df["post"].apply(lambda x: "<mbti>" in x).sum()}")
print(f"sw_df mbti mask counts in augmented col: {sw_df["post_augmented"].apply(lambda x: "<mbti>" in x).sum()}")
print(f"sw_df mbti mask counts in augmented col: {sw_df["post_augmented"].apply(lambda x: "<mbti>" in x).sum()}")

print(f"del_df mbti mask counts in post col (pre aug): {del_df["post"].apply(lambda x: "<mbti>" in x).sum()}")
print(f"del_df mbti mask counts in augmented col: {del_df["post_augmented"].apply(lambda x: "<mbti>" in x).sum()}")

In [ ]:
mask_lost = syn_df["post"].str.contains("<mbti>") & ~syn_df["post_augmented"].str.contains("<mbti>")

failed_masking = pd.DataFrame(syn_df[mask_lost])
#print(mask_lost.sum())

mask_ok = ~(syn_df["post"].str.contains("<mbti>") & ~syn_df["post_augmented"].str.contains("<mbti>"))
syn_df_final = syn_df[mask_ok].reset_index(drop=True)


In [ ]:
mask_lost_sw = ~sw_df["post"].str.contains("<mbti>") & sw_df["post_augmented"].str.contains("<mbti>")
print(sw_df[mask_lost_sw])

mask_lost_del = ~del_df["post"].str.contains("<mbti>") & del_df["post_augmented"].str.contains("<mbti>")
print(del_df[mask_lost_del])

sw_df.at[826, "post_augmented"] = sw_df.at[826, "post_augmented"].replace("<mbti>", "john")
del_df.at[827, "post_augmented"] = del_df.at[827, "post_augmented"].replace("<mbti>", "john")

## Merging the samples together
Since I divided per label I need to put the samples together again and then merge them to the initial dataset.

In [ ]:
# putting them all together
bt_df = bt_df.drop(columns = "post_augmented")
bt_df = bt_df.drop(columns = "post")
bt_df = bt_df.rename(columns={"post_augmented_john": "post"})

syn_df_final = syn_df_final.drop(columns = "post")
syn_df_final = syn_df_final.rename(columns={"post_augmented": "post"})

sw_df = sw_df.drop(columns = "post")
sw_df = sw_df.rename(columns={"post_augmented": "post"})

del_df = del_df.drop(columns = "post")
del_df = del_df.rename(columns={"post_augmented": "post"})

In [ ]:
df_augmented_final = pd.concat([syn_df_final, sw_df, del_df, bt_df])

In [ ]:
df = pd.read_csv("../data/csv/mbti_cleaned.csv")
df = pd.concat([df, df_augmented_final])


In [ ]:
df["label"].value_counts().reset_index()

# Undersampling
To bring all label counts approximately on the same level, label above the chosen threshold must be undersampled. This is simply done by randomly deleting observations.

In [ ]:
from sklearn.utils import resample

def undersample(df, target_col, target_n):
    sampled = []
    for cls in df[target_col].unique():
        cls_df = df[df[target_col] == cls]
        if len(cls_df) > target_n:
            cls_df = resample(cls_df, n_samples=target_n, 
                            random_state=42, replace=False)
        sampled.append(cls_df)
    return pd.concat(sampled).reset_index(drop=True)

df_undersampled = undersample(df, "label", 10000)

In [ ]:
df_undersampled["label"].value_counts().reset_index()

In [ ]:
length_dist = df_undersampled["post"].str.len().describe()
print(length_dist)

In [ ]:
from datasets import ClassLabel, DatasetDict, Dataset

df_undersampled = Dataset.from_pandas(df_undersampled, preserve_index=False)

df_undersampled = df_undersampled.class_encode_column("label")
mbti_labels = ["ENFJ", "ENFP", "ENTJ", "ENTP", "ESFJ", "ESFP", "ESTJ", "ESTP", 
               "INFJ", "INFP", "INTJ", "INTP", "ISFJ", "ISFP", "ISTJ", "ISTP"]


new_features = df_undersampled.features.copy()
new_features["label"] = ClassLabel(names=mbti_labels)

df_undersampled = df_undersampled.cast(new_features)

split = df_undersampled.train_test_split(test_size = 0.2, shuffle = True, stratify_by_column="label", seed = rseed)
train = split["train"]
df_temp = split["test"]

split2 = df_temp.train_test_split(test_size = 0.5, shuffle = True, stratify_by_column= "label", seed = rseed)
val = split2["train"]
test = split2["test"]

df_final = DatasetDict({
    "train": train,
    "test": test,
    "validation": val
})

df_final

df_final.save_to_disk("..\data\mbti_balanced")
df_final.push_to_hub("DrinkIcedT/mbti_balanced")